# State Objective BC Warm Start v1

Goal: bridge the previous state-objective target table into a behavior-cloning / warm-start dataset without opening another broad PPO sweep.

External basis:
- Qlib: reusable RL investment workflow and experiment artifact discipline: https://github.com/microsoft/qlib
- Qlib paper: AI-oriented quantitative investment platform, arXiv: https://arxiv.org/abs/2009.11189
- FinRL: reproducible train / validate / test pipeline for financial DRL: https://github.com/AI4Finance-Foundation/FinRL
- FinRL paper, arXiv: https://arxiv.org/abs/2011.09607
- Offline RL / behavior cloning precedent for stabilizing policy learning before improvement: https://doi.org/10.21314/JCF.2016.322

Governance: this notebook is a no-promotion bridge. The source targets are built from 2020+ OOS and walk-forward evidence, so the output can diagnose representational fit and prepare an interface, but it cannot by itself justify runtime promotion.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "SPEC.md").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.soxl_quant.config import load_profile
from src.soxl_quant.policy.policy_builder import (
    build_state_objective_warm_start_dataset_from_frame,
    state_objective_target_position_from_row,
)

CARD = ROOT / "runtime" / "model_selection" / "card" / "state_objective_bc_warm_start_v1"
TARGETS_PATH = ROOT / "runtime" / "model_selection" / "card" / "state_objective_training_target_v1" / "training_targets.csv"
RECIPE_PATH = ROOT / "runtime" / "model_selection" / "card" / "state_objective_checkpoint_selection_v1" / "training_objective_recipe.json"
PROFILE_NAME = "balanced_rule_residual_v22"
FRAME_PATH = ROOT / "data" / f"soxl_profile_{PROFILE_NAME}.pkl"
CARD.mkdir(parents=True, exist_ok=True)

profile = load_profile(PROFILE_NAME)
targets = pd.read_csv(TARGETS_PATH, parse_dates=["datetime"])
recipe = json.loads(RECIPE_PATH.read_text())
frame = pd.read_pickle(FRAME_PATH)
targets.head()

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


,datetime,candidate,window,cluster,teacher_position,rebound_target_position,risk_target_position,objective_weight,walkforward_relative_weight,downtrend_drawdown_weight,...,upside_position,router_position,mom_5,mom_21,drawdown_63,rv_21,vol_of_vol_21,soxl_soxx_ratio_widen_21,soxl_soxx_ratio_z_252,soxx_mom_21
0,2020-01-02,state_conditioned_rebound_risk_router_v1,strict_oos,NaN,0.0625,0.0625,0.0625,3.331228,2.331228,0.0,...,0.205,0.0625,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
1,2020-01-03,state_conditioned_rebound_risk_router_v1,strict_oos,NaN,0.0625,0.0625,0.0625,3.331228,2.331228,0.0,...,0.205,0.0625,0.0,0.0,-0.054014,0.0,0.0,0.0,0.0,0.0
2,2020-01-06,state_conditioned_rebound_risk_router_v1,strict_oos,NaN,0.0625,0.0625,0.0625,3.331228,2.331228,0.0,...,0.205,0.0625,0.0,0.0,-0.084018,0.0,0.0,0.0,0.0,0.0
3,2020-01-07,state_conditioned_rebound_risk_router_v1,strict_oos,NaN,0.0625,0.0625,0.0625,3.331228,2.331228,0.0,...,0.205,0.0625,0.0,0.0,-0.036786,0.0,0.0,0.0,0.0,0.0
4,2020-01-08,state_conditioned_rebound_risk_router_v1,strict_oos,NaN,0.0625,0.0625,0.0625,3.331228,2.331228,0.0,...,0.205,0.0625,0.0,0.0,-0.034662,0.0,0.0,0.0,0.0,0.0


## Coverage And Leakage Audit

The first gate is whether the existing feature frame overlaps the target dates. The second gate is whether those dates are safe for promotion training. Here they are not: the target table encodes fixed OOS and walk-forward evidence, so this run stays diagnostic.

In [2]:
target_dates = pd.Index(pd.to_datetime(targets["datetime"]).dt.normalize().unique())
frame_dates = pd.Index(pd.to_datetime(frame.index).normalize().unique())
overlap_dates = target_dates.intersection(frame_dates)
duplicate_date_rows = int(len(targets) - targets["datetime"].dt.normalize().nunique())

coverage = {
    "profile": PROFILE_NAME,
    "frame_path": str(FRAME_PATH.relative_to(ROOT)),
    "targets_path": str(TARGETS_PATH.relative_to(ROOT)),
    "target_rows": int(len(targets)),
    "target_unique_dates": int(targets["datetime"].dt.normalize().nunique()),
    "duplicate_date_rows": duplicate_date_rows,
    "frame_rows": int(len(frame)),
    "frame_start": str(pd.Timestamp(frame.index.min()).date()),
    "frame_end": str(pd.Timestamp(frame.index.max()).date()),
    "overlap_dates": int(len(overlap_dates)),
    "overlap_start": str(overlap_dates.min().date()) if len(overlap_dates) else None,
    "overlap_end": str(overlap_dates.max().date()) if len(overlap_dates) else None,
    "promotion_safe": False,
    "promotion_blocker": "targets are derived from 2020+ OOS / walk-forward evidence; use only for no-promotion diagnostic or fold-local target generation",
}
coverage

{'profile': 'balanced_rule_residual_v22',
 'frame_path': 'data/soxl_profile_balanced_rule_residual_v22.pkl',
 'targets_path': 'runtime/model_selection/card/state_objective_training_target_v1/training_targets.csv',
 'target_rows': 2322,
 'target_unique_dates': 1569,
 'duplicate_date_rows': 753,
 'frame_rows': 2973,
 'frame_start': '2011-03-11',
 'frame_end': '2022-12-30',
 'overlap_dates': 756,
 'overlap_start': '2020-01-02',
 'overlap_end': '2022-12-30',
 'promotion_safe': False,
 'promotion_blocker': 'targets are derived from 2020+ OOS / walk-forward evidence; use only for no-promotion diagnostic or fold-local target generation'}

## Target Label Materialization

Per date, the reusable builder keeps the highest `objective_weight` target row. Downtrend risk caps take precedence over rebound floors, then the target position is mapped through the real profile interpreter to produce a valid action label.

In [3]:
targets_for_overlap = targets[targets["datetime"].dt.normalize().isin(overlap_dates)].copy()
targets_for_overlap["state_objective_target_position"] = [
    state_objective_target_position_from_row(row)
    for row in targets_for_overlap.to_dict("records")
]
target_position_summary = (
    targets_for_overlap.groupby(["window", "is_rebound_cluster", "is_downtrend_state"], dropna=False)
    .agg(
        rows=("datetime", "size"),
        avg_teacher_position=("teacher_position", "mean"),
        avg_target_position=("state_objective_target_position", "mean"),
        max_objective_weight=("objective_weight", "max"),
    )
    .reset_index()
    .sort_values(["max_objective_weight", "rows"], ascending=[False, False])
)
target_position_summary

,window,is_rebound_cluster,is_downtrend_state,rows,avg_teacher_position,avg_target_position,max_objective_weight
3,wf_2020,False,True,118,0.142546,0.128217,98.262364
2,wf_2020,False,False,134,0.207288,0.207288,78.569598
1,strict_oos,False,True,478,0.108700,0.094011,8.695242
0,strict_oos,False,False,278,0.253568,0.253568,3.331228
5,wf_2022,False,True,242,0.045999,0.045999,1.000000
4,wf_2022,False,False,8,0.039005,0.039005,1.000000


In [4]:
obs, actions = build_state_objective_warm_start_dataset_from_frame(
    frame,
    targets=targets,
    profile=profile,
    sample_size=0,
)

action_distribution = (
    pd.Series(actions, name="action")
    .value_counts(normalize=False)
    .rename_axis("action")
    .reset_index(name="count")
    .sort_values("action")
)
action_distribution["share"] = action_distribution["count"] / int(len(actions))
action_distribution.to_csv(CARD / "action_distribution.csv", index=False)
target_position_summary.to_csv(CARD / "target_position_summary.csv", index=False)

comparison_rows = []
for comparison_profile_name in ["balanced_rule_residual_v22", "risk_budget_usage_v1", "balanced"]:
    comparison_profile = load_profile(comparison_profile_name)
    comp_obs, comp_actions = build_state_objective_warm_start_dataset_from_frame(
        frame,
        targets=targets,
        profile=comparison_profile,
        sample_size=0,
    )
    comp_dist = pd.Series(comp_actions).value_counts(normalize=True).sort_index()
    entropy = float(-(comp_dist * np.log(comp_dist + 1e-12)).sum())
    comparison_rows.append(
        {
            "profile": comparison_profile_name,
            "obs_dim": int(comp_obs.shape[1]),
            "samples": int(len(comp_actions)),
            "unique_actions": int(comp_dist.shape[0]),
            "dominant_action": int(comp_dist.idxmax()),
            "dominant_action_share": float(comp_dist.max()),
            "action_entropy": entropy,
        }
    )
profile_label_comparison = pd.DataFrame(comparison_rows).sort_values(
    ["dominant_action_share", "action_entropy"],
    ascending=[True, False],
)
profile_label_comparison.to_csv(CARD / "profile_label_comparison.csv", index=False)
best_label_profile = str(profile_label_comparison.iloc[0]["profile"])

dataset_summary = {
    **coverage,
    "obs_shape": [int(v) for v in obs.shape],
    "action_shape": [int(v) for v in actions.shape],
    "action_count": int(len(action_distribution)),
    "dominant_action": int(action_distribution.sort_values("count", ascending=False).iloc[0]["action"]),
    "dominant_action_share": float(action_distribution["share"].max()),
    "label_degeneracy_warning": bool(action_distribution["share"].max() > 0.90),
    "best_label_profile_by_distribution": best_label_profile,
    "profile_label_comparison": profile_label_comparison.to_dict(orient="records"),
    "selected_teacher_candidate": recipe.get("selected_teacher_candidate"),
    "source_decision": recipe.get("objective"),
}
(CARD / "summary.json").write_text(json.dumps(dataset_summary, indent=2, sort_keys=True))
action_distribution

,action,count,share
0,0,733,0.970861
4,1,1,0.001325
2,2,4,0.005298
3,3,1,0.001325
1,4,16,0.021192


## Decision

This run creates a usable reusable interface and verifies that `training_targets.csv` can become profile-valid BC labels. It also shows that direct v22 hybrid-delta BC is label-degenerate, so the next training attempt should prefer a budget-usage target surface or fold-local target generation before PPO.

Next valid optimization step: generate the same state-objective labels inside each walk-forward train fold only, use the helper as a budget-usage BC warm-start before PPO, and retain checkpoints by `checkpoint_objective_loss` plus fixed-split drawdown gates.

In [5]:
selection = {
    "experiment": "state_objective_bc_warm_start_v1",
    "decision": "bc_warm_start_interface_ready_v22_label_degenerate_no_promotion",
    "promotion_allowed": False,
    "promotion_blocker": dataset_summary["promotion_blocker"],
    "profile": PROFILE_NAME,
    "best_label_profile_by_distribution": best_label_profile,
    "label_degeneracy_warning": dataset_summary["label_degeneracy_warning"],
    "dataset_samples": int(len(actions)),
    "next_mechanism": "fold_local_state_objective_budget_usage_bc_then_walkforward_ppo",
    "acceptance_gate_for_next_run": {
        "relative_return_min_pp": 30.0,
        "max_dd_max_pct": 15.0,
        "walkforward_required": True,
        "no_heldout_target_labels_for_training": True,
    },
    "sidecars": [
        "summary.json",
        "action_distribution.csv",
        "profile_label_comparison.csv",
        "target_position_summary.csv",
    ],
}
(CARD / "selection.json").write_text(json.dumps(selection, indent=2, sort_keys=True))
selection

{'experiment': 'state_objective_bc_warm_start_v1',
 'decision': 'bc_warm_start_interface_ready_v22_label_degenerate_no_promotion',
 'promotion_allowed': False,
 'promotion_blocker': 'targets are derived from 2020+ OOS / walk-forward evidence; use only for no-promotion diagnostic or fold-local target generation',
 'profile': 'balanced_rule_residual_v22',
 'best_label_profile_by_distribution': 'risk_budget_usage_v1',
 'label_degeneracy_warning': True,
 'dataset_samples': 755,
 'next_mechanism': 'fold_local_state_objective_budget_usage_bc_then_walkforward_ppo',
 'acceptance_gate_for_next_run': {'relative_return_min_pp': 30.0,
  'max_dd_max_pct': 15.0,
  'walkforward_required': True,
  'no_heldout_target_labels_for_training': True},
 'sidecars': ['summary.json',
  'action_distribution.csv',
  'profile_label_comparison.csv',
  'target_position_summary.csv']}